[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# autocommit and isolation_level &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/stations.db` with the stations, their year of readings and the empty
`audit` table the notebook used. Run it first. Every task opens and closes connections of its own, so
the tasks do not depend on one another, and the last cell removes the scratch folder.


In [1]:
import math
import shutil
import sqlite3
from datetime import datetime, timedelta
from pathlib import Path

SCRATCH = Path("scratch")
SCRATCH.mkdir(exist_ok=True)
DATABASE = SCRATCH / "stations.db"
DATABASE.unlink(missing_ok=True)
STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE stations (id INTEGER PRIMARY KEY, name TEXT NOT NULL, latitude REAL NOT NULL);
    CREATE TABLE readings (id INTEGER PRIMARY KEY, station_id INTEGER NOT NULL REFERENCES stations (id),
                           hour TEXT NOT NULL, celsius REAL);
    CREATE TABLE audit (note TEXT NOT NULL);
""")
ids = {name: build.execute("INSERT INTO stations (name, latitude) VALUES (?, ?)", (name, latitude)).lastrowid
       for name, latitude in LATITUDES.items()}
build.executemany("INSERT INTO readings (station_id, hour, celsius) VALUES (?, ?, ?)",
                  ((ids[station], hour, celsius) for station, hour, celsius in year_of_readings()))
build.commit()
build.close()


def notes_saved(*notes):
    """How many of these notes are saved in audit, seen from a connection of its own."""
    look = sqlite3.connect(DATABASE)
    count = look.execute(f"SELECT COUNT(*) FROM audit WHERE note IN ({', '.join('?' * len(notes))})", notes).fetchone()[0]
    look.close()
    return count


print("built", DATABASE)


built scratch/stations.db


**1.** `isolation_level=None`.


In [2]:
conn = sqlite3.connect(DATABASE, isolation_level=None)
print("autocommit:", conn.autocommit, "| isolation_level:", conn.isolation_level)

conn.execute("INSERT INTO audit VALUES (?)", ("with no BEGIN",))
print("in a transaction after the insert:", conn.in_transaction)
conn.close()

print("saved:", notes_saved("with no BEGIN"))


autocommit: -1 | isolation_level: None
in a transaction after the insert: False
saved: 1


`isolation_level=None` leaves `autocommit` at legacy control, -1, and stops legacy control from
sending `BEGIN`, so the insert ran as a transaction of its own and was saved at once, with no
commit.


**2.** The statements sqlite3 adds.


In [3]:
conn = sqlite3.connect(DATABASE)
conn.set_trace_callback(lambda statement: print("sent:", statement.strip()))

conn.execute("SELECT COUNT(*) FROM readings").fetchone()
conn.execute("INSERT INTO audit VALUES (?)", ("traced",))
conn.commit()
conn.close()


sent: SELECT COUNT(*) FROM readings
sent: BEGIN
sent: INSERT INTO audit VALUES ('traced')
sent: COMMIT


The code wrote the query and the insert. sqlite3 added the `BEGIN` before the insert, since legacy
control begins a transaction before a change to rows, and `commit()` sent the `COMMIT`.


**3.** A `CREATE TABLE` and a rollback, under two settings.


In [4]:
def create_insert_rollback(conn):
    """Create a table, insert a row, roll back, and report whether the table is there and how many rows it has."""
    conn.execute("CREATE TABLE trial (x)")
    conn.execute("INSERT INTO trial VALUES (1)")
    conn.rollback()
    exists = conn.execute("SELECT COUNT(*) FROM sqlite_schema WHERE name = 'trial'").fetchone()[0] == 1
    rows = conn.execute("SELECT COUNT(*) FROM trial").fetchone()[0] if exists else None
    return exists, rows


conn = sqlite3.connect(DATABASE)
print("legacy control:  ", create_insert_rollback(conn))
conn.execute("DROP TABLE trial")
conn.close()

conn = sqlite3.connect(DATABASE, autocommit=False)
print("autocommit=False:", create_insert_rollback(conn))
conn.close()


legacy control:   (True, 0)
autocommit=False: (False, None)


Under legacy control the `CREATE TABLE` ran before any `BEGIN` and was saved, so only the row was
rolled back and the table remained, empty. Under `autocommit=False` the table was created inside the
open transaction, and the rollback removed it.


**4.** Transactions written in SQL.


In [5]:
conn = sqlite3.connect(DATABASE, autocommit=True)

conn.execute("BEGIN")
conn.execute("INSERT INTO audit VALUES (?)", ("first of two",))
conn.execute("INSERT INTO audit VALUES (?)", ("second of two",))
conn.execute("ROLLBACK")
print("saved after ROLLBACK:", notes_saved("first of two", "second of two"))

conn.execute("INSERT INTO audit VALUES (?)", ("no BEGIN",))
conn.rollback()
print("saved after rollback():", notes_saved("no BEGIN"))
conn.close()


saved after ROLLBACK: 0
saved after rollback(): 1


The `ROLLBACK` written in SQL ended the transaction the `BEGIN` began, and threw both notes away. The
insert with no `BEGIN` was saved as it ran, and `rollback()`, which does nothing under
`autocommit=True`, could not change that.


**5.** A moment outside the transaction, for `VACUUM`.


In [6]:
conn = sqlite3.connect(DATABASE, autocommit=False)
print("before:", conn.in_transaction)

conn.autocommit = True
print("during:", conn.in_transaction)
conn.execute("VACUUM")

conn.autocommit = False
print("after: ", conn.in_transaction)
conn.close()


before: True
during: False
after:  True


Setting `autocommit` to `True` committed the open transaction and left the connection outside one,
where `VACUUM` can run, and setting it back to `False` began a new transaction straight away.


**6.** `executescript` inside the open transaction.


In [7]:
conn = sqlite3.connect(DATABASE, autocommit=False)
conn.executescript("""
    INSERT INTO audit VALUES ('scripted one');
    INSERT INTO audit VALUES ('scripted two');
""")
conn.rollback()
conn.close()

print("saved:", notes_saved("scripted one", "scripted two"))


saved: 0


Under `autocommit=False`, `executescript` commits nothing first and adds no transaction of its own,
so both inserts ran inside the open transaction, and the rollback removed them.

Last, remove the scratch folder:


In [8]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [autocommit and isolation_level](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/09-autocommit-and-isolation-level.ipynb)  &nbsp;&middot;&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)
